In [ ]:
import firedrake
from firedrake import Constant, inner, grad, dx, ds_b, derivative
import ufl
import irksome
from irksome import Dt
from zetastokes.terrain_following import free_energy_rate
import numpy as np
from numpy import pi as π
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tqdm.notebook import tqdm, trange

In [ ]:
nx, nz = 80, 16
lx = 5.0
interval = firedrake.IntervalMesh(nx, lx)
mesh = firedrake.ExtrudedMesh(interval, nz)
X, ζ = firedrake.SpatialCoordinate(mesh)
x = X / Constant(lx)

s_0 = Constant(1.0)
δs = Constant(1 / 6)
k = Constant(3.0)
δx = Constant(1 / 7)
b = Constant(0.0)
s = s_0 + δs * firedrake.cos(2 * π * k * (x - δx))

In [ ]:
cg2 = firedrake.FiniteElement("CG", "interval", 2)
r = firedrake.FiniteElement("R", "interval", 0)
h_element = firedrake.TensorProductElement(cg2, r)
H = firedrake.FunctionSpace(mesh, h_element)
b = firedrake.Function(H).interpolate(b)
h = firedrake.Function(H).interpolate(s - b)

In [ ]:
u_element = firedrake.FiniteElement("DQ", "quadrilateral", 2)
p_element = firedrake.FiniteElement("DQ", "quadrilateral", 1)

#Σ = firedrake.TensorFunctionSpace(mesh, τ_element, symmetry=True)
V = firedrake.VectorFunctionSpace(mesh, u_element)
Q = firedrake.FunctionSpace(mesh, p_element)
Z = V * Q

In [ ]:
z = firedrake.Function(Z)
u, p = firedrake.split(z)
fields = {"velocity": u, "pressure": p, "bed": b, "thickness": h}
robin_ids = (1, 2, "bottom")
boundary_data = {"robin_ids": robin_ids}
constants = {"viscosity": 1.0, "gravity": 9.81}

G_flow = free_energy_rate(**fields, **constants, **boundary_data)

μ = Constant(constants["viscosity"])
λ = Constant(0.1)
u_Γ = Constant((0.125, 0.0))
G_friction = 0.5 * μ / λ * inner(u - u_Γ,  u - u_Γ) * ds_b

G = G_flow + G_friction
F = firedrake.derivative(G, z)

In [ ]:
params = {
    "form_compiler_parameters": {"quadrature_degree": 8},
    "solver_parameters": {
        "snes_monitor": ":driven-surface.log",
        #"snes_converged_reason": None,
        "snes_linesearch_type": "nleqerr",
        "snes_linesearch_max_it": 40,
    },
}
firedrake.solve(F == 0, z, **params)

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
ax.set_axis_off()
colors = firedrake.quiver(z.sub(0), axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
W = V * Q * H
w = firedrake.Function(W)
w.sub(0).assign(z.sub(0))
w.sub(1).assign(z.sub(1))
w.sub(2).assign(h);

In [ ]:
u, p, h = firedrake.split(w)
v, q, ϕ = firedrake.TestFunctions(W)

fields = {"velocity": u, "pressure": p, "bed": b, "thickness": h}
robin_ids = (1, 2, "bottom")
boundary_data = {"robin_ids": robin_ids}
constants = {"viscosity": 1.0, "gravity": 9.81}

G_flow = free_energy_rate(**fields, **constants, **boundary_data)

μ = Constant(constants["viscosity"])
λ = Constant(0.1)
u_Γ = Constant((0.125, 0.0))
G_friction = 0.5 * μ / λ * inner(u - u_Γ, u - u_Γ) * ds_b

G = G_flow + G_friction
F_momentum = derivative(G, u, v) + derivative(G, p, q)
F_mass = (Dt(h) * ϕ - inner(h * u, grad(ϕ))) * dx
F = ufl.algorithms.expand_derivatives(F_momentum) + F_mass

In [ ]:
final_time = 10.0
δx = 1 / nx
u_max = np.abs(z.sub(0).dat.data_ro).max()
timestep = 0.5 * δx / u_max
print(f"CFL time: {timestep:g}")
num_steps = int(final_time / timestep)
dt = Constant(final_time / num_steps)
t = Constant(0.0)

method = irksome.BackwardEuler()
solver = irksome.TimeStepper(F, method, t, dt, w, **params)

In [ ]:
ws = [w.copy(deepcopy=True)]
for step in trange(num_steps):
    solver.advance()
    ws.append(w.copy(deepcopy=True))

In [ ]:
H1D = firedrake.FunctionSpace(interval, cg2)
hh = firedrake.Function(H1D)
hhs = []
for w in ws:
    hh.dat.data[:] = w.sub(2).dat.data_ro[:]
    hhs.append(hh.copy(deepcopy=True))

In [ ]:
fig, ax = plt.subplots()
firedrake.plot(hhs[1], axes=ax);

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
ax.set_axis_off()
colors = firedrake.quiver(ws[-10].sub(0), axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
fig, ax = plt.subplots()
ymin, ymax = 0.8, 1.2
ax.set_ylim((ymin, ymax))
firedrake.plot(hhs[0], axes=ax)

def animate(h):
    ax.clear()
    firedrake.plot(h, axes=ax)
    ax.set_ylim((ymin, ymax))

animation = FuncAnimation(fig, animate, tqdm(hhs), interval=1e3/60)

In [ ]:
HTML(animation.to_html5_video())